# K-Means Clustering

En esta lección vamos a aplicar K-Means al dataset de música nigeriana que exploramos antes.

**K-Means** es el algoritmo de clustering más común. Funciona así:
1. Elegís cuántos clusters (k) querés
2. El algoritmo coloca k centroids aleatoriamente
3. Asigna cada punto al centroid más cercano
4. Recalcula los centroids como el promedio de sus puntos
5. Repite hasta que se estabilicen

## 1. Preparar los datos

Primero necesitamos:
- Seleccionar las columnas numéricas para clustering
- Codificar el género como número (LabelEncoder)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Cargar datos (ya limpios de la lección anterior)
df = pd.read_csv("../data/nigerian-songs.csv")
df = df[df['artist_top_genre'] != 'Missing']
df = df[(df['artist_top_genre'] == 'afro dancehall') | (df['artist_top_genre'] == 'afropop') | (df['artist_top_genre'] == 'nigerian pop')]
df = df[(df['popularity'] > 0)]

# Codificar género como número
le = LabelEncoder()
X = df.loc[:, ('artist_top_genre','popularity','danceability','acousticness','loudness','energy')]
y = df['artist_top_genre']
X['artist_top_genre'] = le.fit_transform(X['artist_top_genre'])
y = le.transform(y)

print(f"Dataset: {X.shape[0]} canciones, {X.shape[1]} características")
X.head()

## 2. Visualizar outliers con boxplots

Los boxplots muestran la distribución de cada columna y los valores atípicos (outliers).

**¿Qué observamos?** Hay outliers en varias columnas, pero los mantenemos por ahora para no reducir demasiado el dataset.

In [ ]:
plt.figure(figsize=(20,20), dpi=200)

plt.subplot(4,3,1)
sns.boxplot(x = 'popularity', data = df)

plt.subplot(4,3,2)
sns.boxplot(x = 'acousticness', data = df)

plt.subplot(4,3,3)
sns.boxplot(x = 'energy', data = df)

plt.subplot(4,3,4)
sns.boxplot(x = 'instrumentalness', data = df)

plt.subplot(4,3,5)
sns.boxplot(x = 'liveness', data = df)

plt.subplot(4,3,6)
sns.boxplot(x = 'loudness', data = df)

plt.subplot(4,3,7)
sns.boxplot(x = 'speechiness', data = df)

plt.subplot(4,3,8)
sns.boxplot(x = 'tempo', data = df)

plt.subplot(4,3,9)
sns.boxplot(x = 'time_signature', data = df)

plt.subplot(4,3,10)
sns.boxplot(x = 'danceability', data = df)

plt.subplot(4,3,11)
sns.boxplot(x = 'length', data = df)

plt.subplot(4,3,12)
sns.boxplot(x = 'release_date', data = df)

plt.tight_layout()

## 3. Silhouette Score

El **silhouette score** mide qué tan bien separados están los clusters:
- **Cerca de 1**: cluster denso y bien separado
- **Cerca de 0**: clusters superpuestos
- **Cerca de -1**: puntos asignados al cluster equivocado

Probamos con 3 clusters (porque hay 3 géneros).

In [ ]:
from sklearn.cluster import KMeans
from sklearn import metrics

nclusters = 3
seed = 0

km = KMeans(n_clusters=nclusters, random_state=seed)
km.fit(X)

# Predecir cluster para cada punto
y_cluster_kmeans = km.predict(X)

# Calcular silhouette score
score = metrics.silhouette_score(X, y_cluster_kmeans)
print(f"Silhouette Score: {score:.3f}")
print("\nInterpretación:")
print("- Cerca de 1: clusters bien separados")
print("- Cerca de 0: clusters superpuestos")
print("- Cerca de -1: puntos en cluster equivocado")

## 4. Método del codo (Elbow Method)

¿Cómo saber cuántos clusters usar? El **método del codo** prueba diferentes valores de k y grafica la "inercia" (qué tan compactos son los clusters).

**¿Dónde está el codo?** Donde la curva cambia de pendiente bruscamente — ese es el k óptimo.

**Conceptos clave:**
- **WCSS**: Within-Cluster Sum of Squares — suma de distancias al cuadrado de cada punto a su centroide
- **Inercia**: mide qué tan coherentes son los clusters internamente
- **k-means++**: inicialización inteligente que coloca centroids lejos entre sí

In [ ]:
wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters = i, init = 'k-means++', random_state = 42)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10,5))
sns.lineplot(x=range(1, 11), y=wcss, marker='o', color='red')
plt.title('Elbow Method')
plt.xlabel('Número de clusters')
plt.ylabel('WCSS')
plt.show()

## 5. Visualizar los clusters

Aplicamos K-Means con 3 clusters y graficamos los resultados.

**¿Qué vemos?** Los clusters se superponen bastante — esto confirma que los datos no están muy bien separados.

In [ ]:
kmeans = KMeans(n_clusters = 3)
kmeans.fit(X)
labels = kmeans.predict(X)

plt.figure(figsize=(8,6))
plt.scatter(df['popularity'],df['danceability'],c = labels)
plt.xlabel('Popularity')
plt.ylabel('Danceability')
plt.title('K-Means Clustering (k=3)')
plt.show()

## 6. Evaluar la precisión

Comparamos los clusters predichos con los géneros reales. La precisión no es muy alta — y eso es una lección importante.

In [ ]:
labels = kmeans.labels_
correct_labels = sum(y == labels)

print(f"Resultado: {correct_labels} de {y.size} muestras clasificadas correctamente")
print(f"Precisión: {correct_labels/float(y.size):.2f}")

## ¿Por qué la precisión es baja?

La respuesta está en la **varianza**: los datos están demasiado dispersos y poco correlacionados para formar clusters nítidos.

**Problemas del dataset:**
1. Poca correlación entre características
2. Mucho outliers
3. Los géneros musicales se superponen en estas dimensiones

**¿Qué podrías hacer mejor?**
- Escalar los datos (StandardScaler)
- Usar otras columnas
- Probar otro algoritmo (DBSCAN, Gaussian Mixture)
- Limpiar más los outliers

> 🎓 **Lección clave:** K-Means no es mágico. Si los datos no tienen estructura de cluster natural, el algoritmo va a forzar agrupaciones artificiales.

## Resumen

1. Preparamos el dataset codificando el género como número
2. Visualizamos outliers con boxplots
3. Calculamos silhouette score (~0.53 = mediocre)
4. Usamos el método del codo para elegir k=3
5. Visualizamos clusters superpuestos
6. La precisión fue baja por la varianza de los datos

**Lección principal:** No todos los datasets son buenos candidatos para K-Means. La calidad de los datos importa más que la elección del algoritmo.